# Event-Driven Production Engineering
### Kafka Fundamentals | At-Least-Once | Idempotency | Sagas | Event Sourcing + CQRS

> **System:** ShopFlow -- Kafka backbone, 500k daily users.
> Format: **Mental Model -> Real Scenario -> BEFORE -> AFTER -> Frameworks -> Nuances**

*Shift+Enter to run each cell*

## Setup

In [ ]:
from __future__ import annotations
import uuid
import time
from dataclasses import dataclass, field
from collections import defaultdict
from collections.abc import Callable
print('Setup OK')

---
## 1 · Kafka Fundamentals -- The Retained, Partitioned Log

### Mental Model -- 'The Append-Only Ledger'

```
WHAT   Kafka is a distributed, partitioned, RETAINED log.
       Events are appended; consumers read at their own pace using offsets.
       Unlike queues: events are NOT deleted after consumption.
WHY    Multiple consumer groups can independently read the same events.
       A new service can replay ALL historical events from offset 0.
HOW    Topic -> N partitions -> each partition is an ordered log.
       Producer routes by key: same key -> same partition -> ordering.
       Consumer group: each partition assigned to one consumer.
WHEN   High-throughput event streaming, audit logs, CDC, microservice decoupling.
```

### Nuance 1: Ordering is per-partition, not per-topic
A topic with 4 partitions guarantees order within each partition.
Events for the same order_id all go to the same partition (by order_id key),
so ORDER-specific events are ordered. Across different orders: not ordered.

### Nuance 2: Rebalance = transient duplicate processing
When a consumer joins or leaves, Kafka reassigns partitions (rebalance).
A consumer may have processed events that its successor will re-process.
This is why idempotency is not optional -- rebalances guarantee at-least-once.

### Nuance 3: Offset commit strategy determines delivery guarantee
Auto-commit: offsets committed on a timer. If consumer crashes between commit
and processing, events are skipped (at-most-once). Manual commit AFTER processing
= at-least-once (may re-process on crash, but never skip).

In [ ]:
# Simulated Kafka-like partitioned, retained log

@dataclass
class KafkaEvent:
    key:      str
    value:    dict
    event_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])


class Partition:
    def __init__(self, idx: int):
        self.idx   = idx
        self._log: list[KafkaEvent] = []

    def append(self, event: KafkaEvent) -> int:
        self._log.append(event)
        return len(self._log) - 1   # offset

    def read_from(self, offset: int) -> list[tuple[int, KafkaEvent]]:
        # Retained log -- always replayable from any offset
        return [(i, e) for i, e in enumerate(self._log) if i >= offset]


class Topic:
    def __init__(self, name: str, num_partitions: int = 3):
        self.name       = name
        self.partitions = [Partition(i) for i in range(num_partitions)]

    def publish(self, key: str, event: KafkaEvent) -> tuple[int, int]:
        partition_idx = hash(key) % len(self.partitions)
        offset = self.partitions[partition_idx].append(event)
        return partition_idx, offset


topic = Topic('orders', num_partitions=3)

# Publish 9 events for 3 different orders (key = order_id)
for order_id in ['ord-A', 'ord-B', 'ord-C']:
    for evt_type in ['placed', 'paid', 'shipped']:
        p_idx, offset = topic.publish(
            key=order_id,
            event=KafkaEvent(key=order_id, value={'type': evt_type, 'id': order_id})
        )

print('Partition distribution (same order -> same partition = ordered):')
for i, part in enumerate(topic.partitions):
    events = [(e.key, e.value['type']) for _, e in part.read_from(0)]
    print(f'  Partition {i}: {events}')

print()
print('Replay partition 0 from offset 1 (new service catching up):')
for offset, event in topic.partitions[0].read_from(1):
    print(f'  offset={offset}: {event.key} -> {event.value}')

### Where This Is Seen in Real Frameworks

| Tool | Kafka usage |
|------|-------------|
| **confluent-kafka-python** | `Producer.produce(topic, key=k, value=v)` -- low-level, high-perf |
| **aiokafka** | Async Kafka client for FastAPI/asyncio applications |
| **Faust** | Stream processing framework on top of Kafka |
| **Debezium** | CDC: Postgres/MySQL changes -> Kafka topics automatically |
| **AWS MSK** | Managed Kafka; `KAFKA_AUTO_OFFSET_RESET=earliest` for replay |

---
## 2 · At-Least-Once Delivery + Idempotent Consumer

### Mental Model -- 'Certified Mail That May Arrive Twice'

```
WHAT   At-least-once: the broker guarantees every event is delivered.
       But delivery may happen MORE THAN ONCE (crash-recovery, rebalance).
WHY    Exactly-once delivery is extremely hard to achieve across systems.
       At-least-once + idempotent consumer = exactly-once EFFECT.
HOW    Each event has a unique event_id. Consumer maintains a dedupe store.
       Process -> commit offset. On replay: skip if event_id already seen.
WHEN   Every Kafka consumer. Every webhook handler. Every async task.
```

### Nuance 1: Dedupe store TTL must exceed the max retry window
If Kafka retries for up to 24 hours, the dedupe store (Redis) must keep
event_ids for at least 24 hours. TTL < retry window = undetected duplicates.

### Nuance 2: Exactly-once in Kafka is different from at-least-once + idempotency
Kafka's transactional producer + enable.idempotence=true achieves exactly-once
WITHIN Kafka (producer -> broker). But processing by a consumer + external
side effects (DB writes, emails) still requires consumer-level idempotency.

### Nuance 3: Dead Letter Queue (DLQ) for poison messages
A message that always fails (malformed, causes an exception) will block the
partition forever. After N retries, route to a DLQ (separate topic).
Engineers review the DLQ; system keeps processing.

### Real-World Scenario -- ShopFlow Double Charge Incident

**Incident:** A Kafka consumer processed `order.placed` events and charged the
customer via Stripe. After a deployment, Kafka reset offsets and redelivered
events from the last checkpoint. 1,247 customers were charged twice.

**Root cause:** The consumer had no idempotency check on `event_id`.
**Fix:** Store `event_id` in a Redis set with 48h TTL.
On each event: check Redis first. If seen -> skip and commit offset. If new -> process.

In [ ]:
# At-Least-Once delivery + Idempotent Consumer

@dataclass
class Event:
    event_id: str
    type:     str
    payload:  dict


class IdempotentConsumer:
    def __init__(self, handler: Callable[[Event], None]):
        self._handler      = handler
        self._seen:        set[str] = set()  # in prod: Redis SADD with TTL
        self._offsets:     dict[int, int] = {}
        self.dead_letters: list[Event] = []
        self.processed     = 0
        self.skipped       = 0

    def consume(self, partition_idx: int, offset: int, event: Event,
                max_retries: int = 3) -> None:
        # Skip duplicates
        if event.event_id in self._seen:
            self.skipped += 1
            self._offsets[partition_idx] = offset + 1  # commit and move on
            return

        for attempt in range(1, max_retries + 1):
            try:
                self._handler(event)             # side effect (DB write, API call)
                self._seen.add(event.event_id)   # mark AFTER success
                self._offsets[partition_idx] = offset + 1  # commit offset
                self.processed += 1
                return
            except Exception:
                if attempt == max_retries:
                    self.dead_letters.append(event)  # DLQ
                    self._offsets[partition_idx] = offset + 1  # move past poison msg
                    return


side_effects: list[str] = []

def charge_customer(event: Event) -> None:
    side_effects.append(f'charged-{event.payload["order_id"]}')
    print(f'  [Stripe] Charged {event.payload["order_id"]}')


consumer = IdempotentConsumer(charge_customer)

# Normal delivery
e1 = Event('evt-001', 'order.placed', {'order_id': 'ord-A', 'amount': 49.99})
consumer.consume(0, 0, e1)

# Simulate redelivery (same event_id -- Kafka rebalance)
print('Redelivery after rebalance (same event_id):')
consumer.consume(0, 0, e1)  # should be skipped

# Another unique event
e2 = Event('evt-002', 'order.placed', {'order_id': 'ord-B', 'amount': 99.99})
consumer.consume(0, 1, e2)

print(f'\nProcessed: {consumer.processed}, Skipped: {consumer.skipped}')
print(f'Side effects (charges): {side_effects}')
print('ord-A charged exactly once despite two deliveries')

### Where This Is Seen in Real Frameworks

| Tool | Idempotency mechanism |
|------|----------------------|
| **Redis SETNX** | `redis.set(event_id, 1, ex=172800, nx=True)` -- atomic dedupe |
| **Postgres ON CONFLICT** | `INSERT INTO processed_events ... ON CONFLICT DO NOTHING` |
| **Celery** | `task_always_eager=False` + Redis result backend as dedupe |
| **AWS SQS** | Message deduplication ID on FIFO queues |
| **Kafka idempotent producer** | `enable.idempotence=true` -- exactly-once write to broker |

---
## 3 · Saga -- Distributed Transactions Without 2PC

### Mental Model -- 'Mission Control: Step by Step with Abort Protocol'

```
WHAT   A saga replaces a distributed transaction with a sequence of local
       steps, each with a compensating action (undo) run in reverse on failure.
WHY    2-Phase Commit (2PC) across microservices holds locks across network
       calls: O(n) lock duration, single point of failure (coordinator),
       blocking on slow/down services. Systems don't scale with 2PC.
HOW    Orchestration: a coordinator runs steps and compensations.
       Choreography: each service publishes events; others react.
WHEN   Multi-service workflows: order -> payment -> shipping -> inventory.
```

### Nuance 1: Compensations must be idempotent
If the orchestrator crashes between 'refund' and marking refund done,
it will call refund again on recovery. The refund must be idempotent
(check if already refunded before calling Stripe).

### Nuance 2: Sagas don't provide isolation
Between step 2 (payment charged) and step 5 (order confirmed), another
process can see the charge without a confirmed order. This is normal.
Design UIs and downstream services to handle these intermediate states.

### Nuance 3: Orchestration vs Choreography trade-offs
Orchestration: easier to reason about, single view of workflow, harder to scale.
Choreography: fully decoupled, scales well, hard to debug (distributed control flow).
For complex workflows (> 5 steps), orchestration is usually clearer.

In [ ]:

@dataclass
class SagaStep:
    name:         str
    action:       Callable[[], None]
    compensation: Callable[[], None]


class Saga:
    def __init__(self, name: str):
        self.name   = name
        self._steps: list[SagaStep] = []

    def add(self, step: SagaStep) -> Saga:
        self._steps.append(step); return self

    def execute(self) -> tuple[bool, list[str]]:
        done:  list[SagaStep] = []
        trail: list[str]      = []
        for step in self._steps:
            try:
                step.action()
                done.append(step)
                trail.append(f'  OK   {step.name}')
            except Exception as exc:
                trail.append(f'  FAIL {step.name}: {exc}')
                trail.append('  --- compensating in reverse ---')
                for s in reversed(done):     # undo in reverse order
                    s.compensation()
                    trail.append(f'  UNDO {s.name}')
                return False, trail
        return True, trail


@dataclass
class OrderSystem:
    order_placed:     bool = False
    payment_charged:  bool = False
    inventory_held:   bool = False
    shipped:          bool = False
    fail_at:          str  = ''

    def build_saga(self) -> Saga:
        saga = Saga('checkout')
        saga.add(SagaStep(
            'place_order',
            lambda: setattr(self, 'order_placed', True),
            lambda: setattr(self, 'order_placed', False),
        ))
        saga.add(SagaStep(
            'charge_payment',
            lambda: self._charge(),
            lambda: setattr(self, 'payment_charged', False),
        ))
        saga.add(SagaStep(
            'hold_inventory',
            lambda: self._hold_inv(),
            lambda: setattr(self, 'inventory_held', False),
        ))
        saga.add(SagaStep(
            'ship',
            lambda: self._ship(),
            lambda: setattr(self, 'shipped', False),
        ))
        return saga

    def _charge(self):
        if self.fail_at == 'payment': raise RuntimeError('Card declined')
        self.payment_charged = True

    def _hold_inv(self):
        if self.fail_at == 'inventory': raise RuntimeError('Out of stock')
        self.inventory_held = True

    def _ship(self):
        if self.fail_at == 'ship': raise RuntimeError('Warehouse error')
        self.shipped = True


# Happy path
sys1 = OrderSystem()
ok, trail = sys1.build_saga().execute()
print(f'Happy path (ok={ok}):')
for line in trail: print(line)
print(f'  State: {sys1}')

# Failure at inventory -- payment must be refunded
print('\nInventory failure (payment should be refunded):')
sys2 = OrderSystem(fail_at='inventory')
ok, trail = sys2.build_saga().execute()
for line in trail: print(line)
print(f'  payment_charged={sys2.payment_charged} (should be False -- refunded)')

### Where This Is Seen in Real Frameworks

| Tool | Saga implementation |
|------|--------------------|
| **Temporal** | Durable workflow engine with built-in saga + compensation |
| **Conductor (Netflix)** | Microservice orchestration workflow engine |
| **AWS Step Functions** | State machine with retry + catch -> compensation |
| **Celery chains/chords** | Chain of tasks with error callbacks for compensation |
| **MassTransit (.NET)** | Built-in saga state machine with persistent state |

---
## 4 · Event Sourcing + CQRS -- The Ledger Approach

### Mental Model -- 'Bank Ledger vs Bank Balance'

```
WHAT   Event Sourcing: store every state CHANGE as an event (never mutate).
       CQRS: separate Read and Write models.
       State = apply(all events from the beginning).
WHY    You get: full audit trail, time travel (state at any past moment),
       event replay to build new read models.
HOW    Event store: append-only list of events per aggregate.
       Read model (projection): built by replaying events.
       Command -> event -> projection update.
WHEN   Audit requirements, complex domain logic, financial systems,
       systems where 'what happened' matters as much as 'what is now'.
```

### Nuance 1: Snapshot optimization for long-lived aggregates
An order with 10,000 events takes 10,000 replays to load.
Snapshot: every N events, store a checkpoint of the current state.
Load = load latest snapshot + replay events since snapshot.

### Nuance 2: Event schema versioning is HARD
Events are immutable and stored forever. If you rename a field, old events
have the old name. Solutions: upcasting (transform old events on read),
versioned event types (`OrderPlacedV1`, `OrderPlacedV2`), copy-on-write.

### Nuance 3: CQRS read models are eventually consistent
A command (write) triggers an event. The projection (read model) is updated
asynchronously. A user who writes then immediately reads may see stale data.
Design UIs to handle this (optimistic updates, 'Your order is being confirmed...').

In [ ]:
# Event Sourcing + CQRS

@dataclass
class DomainEvent:
    event_id:   str
    event_type: str
    payload:    dict
    occurred_at:float = field(default_factory=time.time)


class EventStore:
    def __init__(self):
        self._streams: dict[str, list[DomainEvent]] = defaultdict(list)

    def append(self, aggregate_id: str, event: DomainEvent) -> None:
        self._streams[aggregate_id].append(event)

    def load(self, aggregate_id: str, since_version: int = 0) -> list[DomainEvent]:
        return self._streams[aggregate_id][since_version:]

    def all_events(self) -> list[DomainEvent]:
        events = [e for stream in self._streams.values() for e in stream]
        return sorted(events, key=lambda e: e.occurred_at)


# Write model: Order aggregate rebuilt from events
class Order:
    def __init__(self, order_id: str):
        self.order_id = order_id
        self.status   = 'empty'
        self.total    = 0.0
        self.history: list[str] = []
        self._version = 0

    def apply(self, event: DomainEvent) -> None:
        if event.event_type == 'order.placed':
            self.status = 'placed'
            self.total  = event.payload['total']
        elif event.event_type == 'order.paid':
            self.status = 'paid'
        elif event.event_type == 'order.shipped':
            self.status = 'shipped'
        self.history.append(event.event_type)
        self._version += 1

    @classmethod
    def from_events(cls, order_id: str, events: list[DomainEvent]) -> Order:
        order = cls(order_id)
        for e in events: order.apply(e)
        return order


# Read model: projection for the orders dashboard
class OrdersDashboard:
    def __init__(self):
        self.by_status: dict[str, list[str]] = defaultdict(list)
        self.revenue_by_day: dict[str, float] = defaultdict(float)

    def handle(self, event: DomainEvent) -> None:
        if event.event_type == 'order.placed':
            self.by_status['placed'].append(event.payload['order_id'])
            day = '2026-08-15'  # simplified
            self.revenue_by_day[day] += event.payload.get('total', 0)
        elif event.event_type == 'order.paid':
            oid = event.payload['order_id']
            if oid in self.by_status.get('placed', []):
                self.by_status['placed'].remove(oid)
            self.by_status['paid'].append(oid)


store     = EventStore()
dashboard = OrdersDashboard()

# Commands generate events
events_to_emit = [
    DomainEvent('e1', 'order.placed',  {'order_id': 'ord-A', 'total': 49.99}),
    DomainEvent('e2', 'order.placed',  {'order_id': 'ord-B', 'total': 99.99}),
    DomainEvent('e3', 'order.paid',    {'order_id': 'ord-A'}),
    DomainEvent('e4', 'order.shipped', {'order_id': 'ord-A'}),
]

for evt in events_to_emit:
    store.append(evt.payload['order_id'], evt)
    dashboard.handle(evt)   # update read model

# Rebuild order state from events (Write model)
order_a = Order.from_events('ord-A', store.load('ord-A'))
print(f'Order A state: status={order_a.status}, total={order_a.total}')
print(f'Order A history: {order_a.history}')

# Query read model (CQRS split)
print(f'Dashboard placed:  {dashboard.by_status["placed"]}')
print(f'Dashboard paid:    {dashboard.by_status["paid"]}')
print(f'Revenue 2026-08-15: ${dashboard.revenue_by_day["2026-08-15"]:.2f}')

# Time travel: what was Order A's state after only the first 2 events?
order_a_v1 = Order.from_events('ord-A', store.load('ord-A', since_version=0)[:1])
print(f'\nTime travel (after only event 1): status={order_a_v1.status}')

### Where This Is Seen in Real Frameworks

| Tool | Event Sourcing / CQRS |
|------|----------------------|
| **EventStoreDB** | Purpose-built event store with projections |
| **Marten (C#)** | Postgres as event store + projection engine |
| **Axon Framework** | Java/Kotlin framework for ES+CQRS with Saga |
| **Kafka + Faust** | Kafka as event store; Faust builds read models via stream processing |
| **sourced (Python)** | Lightweight ES library for Python domain models |

---
## 5 · Dead Letter Queue -- Handling Poison Messages

### Mental Model -- 'The Returns Desk at the Post Office'

```
WHAT   A separate queue for messages that fail processing after N retries.
WHY    A malformed or unexpected message that always fails will block
       partition processing indefinitely (or until offset is manually reset).
       The DLQ isolates poison messages; the main queue keeps flowing.
HOW    After max_retries, publish to <topic>.DLQ instead of committing offset.
       Commit offset on the main topic (move past the poison message).
       Engineers review DLQ messages; fix code; replay to main topic.
WHEN   Any event consumer. Essential for at-least-once consumers.
```

### Nuance 1: DLQ messages must include failure metadata
Store with the DLQ message: `original_topic`, `original_offset`, `partition`,
`error_message`, `attempt_count`, `first_failure_at`.
Without this, engineers can't diagnose why the message failed.

### Nuance 2: DLQ replay must be rate-limited
Replaying 10,000 DLQ messages at full speed after a fix can cause a burst
that re-overwhelms the fixed service. Replay with throttling.

### Nuance 3: Monitor DLQ depth as a key metric
A growing DLQ depth = something is broken. Alert on DLQ depth > 0 for
critical topics (payments, orders). DLQ is a canary for schema changes
and unhandled exception types.

In [ ]:
# Dead Letter Queue implementation

@dataclass
class DLQMessage:
    original_event:  dict
    error_message:   str
    attempt_count:   int
    original_topic:  str
    original_offset: int
    failed_at:       float = field(default_factory=time.time)


class ConsumerWithDLQ:
    def __init__(self, handler: Callable[[dict], None],
                 max_retries: int = 3):
        self._handler    = handler
        self.max_retries = max_retries
        self.dlq:        list[DLQMessage] = []
        self.processed   = 0
        self.dlq_count   = 0

    def consume(self, topic: str, offset: int, event: dict) -> str:
        last_error = ''
        for attempt in range(1, self.max_retries + 1):
            try:
                self._handler(event)
                self.processed += 1
                return 'processed'
            except Exception as exc:
                last_error = str(exc)
                print(f'  Attempt {attempt}/{self.max_retries} failed: {exc}')

        # All retries exhausted -> DLQ
        self.dlq.append(DLQMessage(
            original_event=event,
            error_message=last_error,
            attempt_count=self.max_retries,
            original_topic=topic,
            original_offset=offset,
        ))
        self.dlq_count += 1
        print(f'  Sent to DLQ after {self.max_retries} retries')
        return 'dead_lettered'


def fragile_handler(event: dict) -> None:
    if event.get('corrupt'): raise ValueError('Malformed event: missing required field')
    print(f'  Processed: {event["type"]}')


consumer = ConsumerWithDLQ(fragile_handler, max_retries=3)

events = [
    {'type': 'order.placed', 'order_id': 'ord-A'},
    {'type': 'order.placed', 'order_id': 'ord-B', 'corrupt': True},  # poison
    {'type': 'order.placed', 'order_id': 'ord-C'},
]

for i, evt in enumerate(events):
    print(f'\nProcessing offset {i}:')
    result = consumer.consume('orders', i, evt)
    print(f'  -> {result}')

print(f'\nResults: processed={consumer.processed}, dlq={consumer.dlq_count}')
print('Main queue kept flowing despite poison message at offset 1')

print('\nDLQ contents (for engineer review):')
for msg in consumer.dlq:
    print(f'  topic={msg.original_topic} offset={msg.original_offset} '
          f'error={msg.error_message!r}')

### Where This Is Seen in Real Frameworks

| Tool | DLQ implementation |
|------|-------------------|
| **AWS SQS** | Every queue can have a DLQ; set `maxReceiveCount=3` |
| **Kafka** | Convention: `<topic>.DLQ` topic; produce failed events there |
| **Celery** | `task_reject_on_worker_lost=True` + `max_retries=3` + fallback handler |
| **RabbitMQ** | `x-dead-letter-exchange` header routes failed messages to DLQ exchange |
| **Redpanda** | Same Kafka protocol; DLQ is an application-level convention |